## Finalizing labels

Produce the unified labels and assemble the gold dataset: take the majority topic (leaving topic blank only when all three annotators disagree) and the consensus priority, write `final_annotation_dataset.csv`, then merge with the email text to build `gold_dataset.csv` (the input to preprocessing).

In [1]:
import numpy as np
import pandas as pd

In [6]:
# Input
in_path = "../data/annotation/annotation_dataset_clean_3label.csv"
out_path = "../data/annotation/final_annotation_dataset.csv"

df = pd.read_csv(in_path)

topic_cols = ["ge_topic_label", "cl_topic_label", "gpt_topic_label"]
priority_col = "priority_consensus_label"

# Normalize topic labels for comparison (case/space-insensitive)
topics = df[topic_cols].apply(lambda s: s.astype("string").str.strip().str.lower())
t = topics.fillna("__NA__")

# Disagreement rule: ALL three labels differ from each other (no majority).
# (2-of-3 majority is NOT a disagreement -- the majority label is taken as unified.)
mismatch_count = (t[topic_cols[0]] != t[topic_cols[1]]).astype(int) + (t[topic_cols[0]] != t[topic_cols[2]]).astype(int) + (t[topic_cols[1]] != t[topic_cols[2]]).astype(int)
disagree = mismatch_count == 3

# Show disagreements (rows needing manual review)
disagreements_df = df.loc[disagree, ["id", *topic_cols]]
disagreements_df

# Unified topic label: majority vote when not in strong disagreement
majority = t.mode(axis=1)[0].replace("__NA__", pd.NA)
unified_topic = pd.Series(pd.NA, index=df.index, dtype="string")
unified_topic.loc[~disagree] = majority.loc[~disagree]

final_df = pd.DataFrame(
    {
        "id": df["id"],
        "topic": unified_topic,
        "priority": df[priority_col],
    }
)

final_df.to_csv(out_path, index=False)
final_df.head()

,id,topic,priority
0,16f6a7b56965be20,deadline-action,Medium
1,1788fbf21ce98f4f,administrative,Low
2,17c95e93525c1d80,course-exam,Medium
3,1d22187c26688ffa,course-exam,Medium
4,26ba4f21835dd402,administrative,Low


In [ ]:
import json
import pandas as pd

labels = pd.read_csv("../data/annotation/final_annotation_dataset.csv", dtype={"id": "string"})
emails = pd.read_json("../data/raw/filtered_emails.jsonl", lines=True, dtype={"id": "string"})

# === APPENDIX (optional, beyond report scope): list emails that did not receive a label, for manual inspection ===
missing = emails.loc[~emails["id"].isin(labels["id"])]
for msg in missing.to_dict(orient="records"):
    print(json.dumps(msg, ensure_ascii=False))

gold = emails.merge(labels, on="id", how="inner")[["id", "subject", "body_plain", "topic", "priority"]]
gold.to_csv("../data/processed/gold_dataset.csv", index=False)

gold.head()

In [8]:
gold = pd.read_csv("../data/processed/gold_dataset.csv", dtype={"id": "string"})

print(f"Total rows       : {len(gold)}")
print(f"Missing topic    : {gold['topic'].isna().sum()}")
print(f"Missing priority : {gold['priority'].isna().sum()}")
print()

# Show a sample of rows that have at least one missing label
broken = gold[gold["topic"].isna() | gold["priority"].isna()]
print(f"Rows with any missing label: {len(broken)}")
broken.head(10)

Total rows       : 919
Missing topic    : 0
Missing priority : 0

Rows with any missing label: 0


,id,subject,body_plain,topic,priority
